# Theme 2: RAG Application, Starter Kit

This starter kit contains a complete, working **Retrieval-Augmented Generation (RAG)** application using **Gemma 4**, Google's open-weight model. Your team's task is to adapt and extend it for the problem statement you selected (2A or 2B).

**How RAG works:**

```
 Document ──► 1. split into chunks ──► 2. convert each chunk into an embedding (a list of numbers)
                                                           │
 Question ──► 3. retrieve the chunks whose embeddings are closest to the question's embedding
                                                           │
              4. give ONLY those chunks to Gemma ──► answer with page citations
```

**Models used (both open-weight):**
- `all-MiniLM-L6-v2` (sentence-transformers) creates the embeddings. It runs inside this notebook.
- `gemma-4-26b-a4b-it` (Google) writes the answers. It is accessed through the free Google AI Studio API.

**No GPU is required.** Use the default Colab runtime (CPU).

### Obtaining a free Google AI Studio API key (one per team)
1. Go to **https://aistudio.google.com/apikey** and sign in with a Google account.
2. Select **Create API key** and copy the key.
3. In Colab, select the **key icon** in the left sidebar (**Secrets**).
4. Select **Add new secret**. Set **Name** to `GOOGLE_API_KEY` and **Value** to your key. Enable **Notebook access**.

> **Important:** do not paste the key directly into a code cell. Anyone the notebook is shared with would be able to see it.
>
> **No key yet:** the notebook still runs. The retrieval stages work normally, and the application shows the passages it found in place of a written answer.

### How to use this notebook
- Run the cells **in order, from top to bottom**, the first time (select a cell and press **Shift + Enter**).
- Comments marked **STUDENT TODO** show the intended places to make changes.
- After changing a cell, re-run that cell and any cells below it that depend on it.
- If the notebook stops working as expected, select **Runtime → Restart session** and run all cells again from the top.

## Step 0: Installation (run once)

In [2]:
# Installs the required libraries. This takes about one minute.
!pip install -q pypdf sentence-transformers "gradio>=6,<7" google-genai

## Step 1: Settings

These values control how documents are divided, searched and answered. They are the main parameters to experiment with.

In [3]:
# Scrapping Required data for model training

import re, shutil, subprocess, tempfile
from pathlib import Path

# 1. repo links go here
REPOS = [
    "https://github.com/firstcontributions/first-contributions",
    "https://github.com/gradio-app/gradio",
]

OUT_DIR = Path("docs")

# 2. Which files count as "contributor docs" (case-insensitive name match)
NAME_PATTERNS = [
    r"^readme", r"^contributing", r"^code_of_conduct", r"^development",
    r"^setup", r"^hacking", r"^style", r"^pull_request_template",
    r"^getting.?started", r"^install",
]
EXTENSIONS = {".md", ".txt", ".rst"}
MAX_DEPTH = 3          # how deep into folders to look
MAX_FILES_PER_REPO = 12
SKIP_DIRS = {"node_modules", ".git", "vendor", "tests", "test", "examples"}

def repo_name(url):
    return url.rstrip("/").removesuffix(".git").split("/")[-1]

def is_doc(path: Path):
    return (path.suffix.lower() in EXTENSIONS and
            any(re.match(p, path.stem.lower()) for p in NAME_PATTERNS))

def collect(url):
    name = repo_name(url)
    dest = OUT_DIR / name
    dest.mkdir(parents=True, exist_ok=True)

    with tempfile.TemporaryDirectory() as tmp:
        subprocess.run(["git", "clone", "--depth", "1", url, tmp],
                       check=True, capture_output=True)
        root = Path(tmp)
        found = []
        for p in root.rglob("*"):
            rel = p.relative_to(root)
            if (p.is_file() and len(rel.parts) <= MAX_DEPTH
                    and not (set(rel.parts) & SKIP_DIRS) and is_doc(p)):
                found.append(p)

        # shortest paths first, so top-level files win
        found.sort(key=lambda p: len(p.relative_to(root).parts))
        for p in found[:MAX_FILES_PER_REPO]:
            rel = p.relative_to(root)
            # flatten subfolders into the name so nothing collides
            flat = "_".join(rel.parts)
            target = dest / f"{name}_{flat}"
            shutil.copy(p, target)
            print("saved", target)

for url in REPOS:
    try:
        collect(url)
    except subprocess.CalledProcessError as e:
        print("Failed:", url, e.stderr.decode()[:200])


saved docs/first-contributions/first-contributions_README.md
saved docs/first-contributions/first-contributions_.github_PULL_REQUEST_TEMPLATE.md
saved docs/first-contributions/first-contributions_.github_CODE_OF_CONDUCT.md
saved docs/first-contributions/first-contributions_.github_CONTRIBUTING.md
saved docs/first-contributions/first-contributions_docs_translations_README.slk.md
saved docs/first-contributions/first-contributions_docs_translations_README.bg.md
saved docs/first-contributions/first-contributions_docs_translations_README.tr.md
saved docs/first-contributions/first-contributions_docs_translations_README.lv.md
saved docs/first-contributions/first-contributions_docs_translations_README.hau.md
saved docs/first-contributions/first-contributions_docs_translations_README.yor.md
saved docs/first-contributions/first-contributions_docs_translations_README.mk.md
saved docs/first-contributions/first-contributions_docs_translations_README.mr.md
saved docs/gradio/gradio_README.md
saved do

In [5]:
GEMMA_MODEL = "gemma-4-26b-a4b-it"   # keep the fast one while developing

CHUNK_SIZE = 150          # words
CHUNK_OVERLAP = 30        # words
TOP_K = 4

ANSWER_RULES = """You are a helpful guide for newcomers who want to contribute to an open-source project.
You answer questions using ONLY the context passages provided.
- Each passage begins with its source file name, for example [CONTRIBUTING.md].
- After each fact you use, cite the file in brackets, for example (CONTRIBUTING.md).
- If the answer is not in the context, reply exactly: "The answer could not be found in the documentation." Then suggest checking the project's issue tracker or asking the maintainers.
- For setup or process questions, give numbered steps. Otherwise keep answers to 2 to 4 sentences.
- Never use outside knowledge."""

TEMPERATURE = 0.2

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

## Step 2: Connect to Gemma and load the embedding model

In [6]:
import os
from google import genai
from google.genai import types, errors


def _get_secret(name):
    """Reads a secret from the Colab Secrets panel (or from an environment variable)."""
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)


GOOGLE_API_KEY = _get_secret("GOOGLE_API_KEY")
DEMO_MODE = not GOOGLE_API_KEY
client = None if DEMO_MODE else genai.Client(api_key=GOOGLE_API_KEY)


def _friendly_error(e):
    code = getattr(e, "code", None)
    if code in (400, 401, 403) and "key" in str(e).lower():
        return "Error: the GOOGLE_API_KEY was rejected. Check that the complete key is saved in Colab Secrets, then re-run this cell."
    if code == 429:
        return "Error: the free-tier usage limit has been reached. Wait about one minute and try again."
    if code == 404:
        return f"Error: the model '{GEMMA_MODEL}' was not found. Check the GEMMA_MODEL setting."
    return f"Error: the request to Gemma failed ({e})."


def generate(user_text, system_text=None, temperature=TEMPERATURE, demo_reply="(Demonstration mode response)"):
    """Sends one message to Gemma and returns its reply as text.
    In demonstration mode, returns `demo_reply`. If the request fails, returns a message beginning with "Error:"."""
    if DEMO_MODE:
        return demo_reply
    try:
        response = client.models.generate_content(
            model = GEMMA_MODEL,
            contents = user_text,
            config = types.GenerateContentConfig(
                system_instruction = system_text,
                temperature = temperature,
                thinking_config = types.ThinkingConfig(thinking_level="minimal"),  # faster replies
            ),
        )
        return (response.text or "").strip()
    except errors.APIError as e:
        return _friendly_error(e)
    except Exception as e:
        return f"Error: {e}"


if DEMO_MODE:
    print("Demonstration mode: no GOOGLE_API_KEY was found. Retrieval will work; answers will show the passages found.")
else:
    print(f"Connected to Google AI Studio. Model: {GEMMA_MODEL}")
    print(generate("In one sentence, what is an open-weight AI model?"))

from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer(EMBED_MODEL_NAME)   # runs on the CPU
print(f"Loaded embedding model: {EMBED_MODEL_NAME}")

Connected to Google AI Studio. Model: gemma-4-26b-a4b-it
Error: the request to Gemma failed (500 INTERNAL. {'error': {'code': 500, 'message': 'Internal error encountered.', 'status': 'INTERNAL'}}).


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loaded embedding model: sentence-transformers/all-MiniLM-L6-v2


## Step 3: Sample document
A short, fictional course syllabus is provided so the application works immediately. Each item in the list is one page. Your own PDF can be uploaded in the application in Step 5.

In [7]:
from pathlib import Path

DOCS_DIR = Path("docs")          # the folder from your clone script
EXTENSIONS = {".md", ".txt"}

SAMPLE_PAGES = []     # the text of each file (keeps the kit's variable name)
PAGE_SOURCES = []     # the file name for each entry, used for citations

for path in sorted(DOCS_DIR.rglob("*")):
    if path.suffix.lower() in EXTENSIONS:
        text = path.read_text(encoding="utf-8", errors="ignore").strip()
        if text:
            SAMPLE_PAGES.append(text)
            PAGE_SOURCES.append(path.name)

print(f"Loaded {len(SAMPLE_PAGES)} files:")
for name in PAGE_SOURCES:
    print(" -", name)

Loaded 24 files:
 - first-contributions_.github_CODE_OF_CONDUCT.md
 - first-contributions_.github_CONTRIBUTING.md
 - first-contributions_.github_PULL_REQUEST_TEMPLATE.md
 - first-contributions_README.md
 - first-contributions_docs_translations_README.bg.md
 - first-contributions_docs_translations_README.hau.md
 - first-contributions_docs_translations_README.lv.md
 - first-contributions_docs_translations_README.mk.md
 - first-contributions_docs_translations_README.mr.md
 - first-contributions_docs_translations_README.slk.md
 - first-contributions_docs_translations_README.tr.md
 - first-contributions_docs_translations_README.yor.md
 - gradio_.changeset_README.md
 - gradio_.github_PULL_REQUEST_TEMPLATE.md
 - gradio_CODE_OF_CONDUCT.md
 - gradio_CONTRIBUTING.md
 - gradio_README.md
 - gradio_client_js_README.md
 - gradio_client_python_README.md
 - gradio_gradio_icons_README.md
 - gradio_guides_CONTRIBUTING.md
 - gradio_js_README.md
 - gradio_readme_template.md
 - gradio_style.md


## Step 4: The RAG pipeline
Each function is one stage of the pipeline. This is the core of the project and the main code to extend.

In [10]:
import numpy as np
from pypdf import PdfReader

knowledge_base = {"name": None, "chunks": [], "vectors": None}

# Below this best-match score, we refuse without calling Gemma. Tune it later (see below).
MIN_SCORE = 0.30
REFUSAL = ("The answer could not be found in the documentation. "
           "Try checking the project's issue tracker or asking the maintainers.")


def read_pdf(path):
    """Returns a list of page texts from a PDF file (kept so PDFs still work)."""
    return [page.extract_text() or "" for page in PdfReader(path).pages]


def split_into_chunks(texts, sources):
    """STAGE 1: split each file into overlapping chunks of about CHUNK_SIZE words."""
    chunks = []
    step = max(1, CHUNK_SIZE - CHUNK_OVERLAP)
    for source, text in zip(sources, texts):
        words = text.split()
        for start in range(0, len(words), step):
            chunk_words = words[start:start + CHUNK_SIZE]
            if len(chunk_words) > 5:  # skip very short fragments
                chunks.append({
                    "source": source,
                    "page": source,   # keeps the later Gradio cell working until we update it
                    "text": " ".join(chunk_words),
                })
            if start + CHUNK_SIZE >= len(words):
                break
    return chunks


def build_knowledge_base(texts, sources, name):
    """STAGE 2: convert each chunk into an embedding."""
    chunks = split_into_chunks(texts, sources)
    if not chunks:
        raise ValueError("No text was found. Check that your docs folder has .md or .txt files.")
    vectors = embedder.encode([c["text"] for c in chunks], normalize_embeddings=True)
    knowledge_base.update(name=name, chunks=chunks, vectors=vectors)
    return f"Loaded **{name}**: {len(texts)} files, {len(chunks)} chunks."


def retrieve(question, top_k=TOP_K):
    """STAGE 3: find the chunks whose meaning is closest to the question."""
    question_vector = embedder.encode([question], normalize_embeddings=True)[0]
    similarities = knowledge_base["vectors"] @ question_vector   # 1.0 = identical meaning
    best = np.argsort(similarities)[::-1][:top_k]
    return [{**knowledge_base["chunks"][i], "score": float(similarities[i])} for i in best]


def answer(question):
    """STAGE 4: give the retrieved chunks to Gemma and ask it to answer using only them."""
    found = retrieve(question)

    # Relevance threshold: nothing close enough, so decline without calling the model.
    if found[0]["score"] < MIN_SCORE:
        return REFUSAL, found

    context = "\n\n".join(f"[{c['source']}] {c['text']}" for c in found)
    prompt = f"Context:\n{context}\n\nQuestion: {question}"
    demo = (f"**Demonstration mode:** no API key is set, so Gemma did not write an answer. Retrieval completed; "
            f"the most relevant passage is in **{found[0]['source']}**:\n\n> {found[0]['text'][:400]}...")
    reply = generate(prompt, system_text=ANSWER_RULES, demo_reply=demo)
    return reply, found


# Quick test with YOUR documents (run the loading cell first).
print(build_knowledge_base(SAMPLE_PAGES, PAGE_SOURCES, "Contributor docs"))
test_question = "How do I submit a pull request?"
reply, found = answer(test_question)
print(f"\nQuestion: {test_question}\nAnswer: {reply}\n\nRetrieved passages:")
for c in found:
    print(f"   [{c['source']}, similarity {c['score']:.2f}] {c['text'][:90]}...")

Loaded **Contributor docs**: 24 files, 157 chunks.

Question: How do I submit a pull request?
Answer: 1. Go to your repository on GitHub and click the `Compare & pull request` button [first-contributions_README.md].
2. Submit the pull request [first-contributions_README.md].

Retrieved passages:
   [first-contributions_docs_translations_README.slk.md, similarity 0.45] pull request" /> Teraz predlož požiadavku na vytiahnutie. <img style="float: right;" src="...
   [first-contributions_README.md, similarity 0.44] looks anything like this: <pre>origin https://github.com/your-username/your_repo.git (fetc...
   [first-contributions_docs_translations_README.hau.md, similarity 0.39] ku. ## Tura duk waɗannan canje-canje zuwa GitHub Yi canje-canjen ku ta amfani da umarnin '...
   [first-contributions_.github_PULL_REQUEST_TEMPLATE.md, similarity 0.39] Before submitting this pull request, check the changes to see it's only the changes you ma...


## Step 5: Launch the application
Run this cell to open the application below the cell and create a **public link** (`https://....gradio.live`). Re-run it after changing any cell above.

In [11]:
import os
import gradio as gr

gr.close_all()  # closes any previous version of the application that is still running

APP_TITLE = "Open-Source Contributor Guide"
APP_DESCRIPTION = ("Ask how to contribute to the project. Answers are based only on the project's "
                   "documentation and cite the source file.")

CHECKLIST_RULES = """You help first-time open-source contributors get started.
Using ONLY the context passages provided, write a numbered step-by-step checklist for making a first contribution
(set up, find an issue, make a change, submit a pull request).
- After each step, cite the file in brackets, for example (CONTRIBUTING.md).
- If the context does not cover a step, skip it. Do not invent steps.
- Keep each step to one line."""


def on_upload(files):
    if not files:
        return "No files uploaded."
    try:
        texts, sources = [], []
        for f in files:
            name = os.path.basename(f)
            if name.lower().endswith(".pdf"):
                for i, page in enumerate(read_pdf(f), start=1):
                    if page.strip():
                        texts.append(page)
                        sources.append(f"{name} p.{i}")
            else:
                text = open(f, encoding="utf-8", errors="ignore").read().strip()
                if text:
                    texts.append(text)
                    sources.append(name)
        return build_knowledge_base(texts, sources, f"{len(files)} uploaded file(s)")
    except Exception as e:
        raise gr.Error(f"The files could not be read: {e}")


def format_sources(found):
    return "\n\n".join(
        f"**{c['source']}** (similarity {c['score']:.2f})\n> {c['text']}" for c in found
    )


def on_ask(question):
    if not question.strip():
        raise gr.Error("Enter a question first.")
    reply, found = answer(question)
    return reply, format_sources(found)


def on_checklist():
    found = retrieve("set up the project, fork, clone, create a branch, make changes, submit a pull request",
                     top_k=6)
    context = "\n\n".join(f"[{c['source']}] {c['text']}" for c in found)
    prompt = f"Context:\n{context}\n\nWrite the onboarding checklist."
    demo = "**Demonstration mode:** no API key is set, so the checklist could not be generated."
    reply = generate(prompt, system_text=CHECKLIST_RULES, demo_reply=demo)
    return reply, format_sources(found)


with gr.Blocks(title=APP_TITLE) as demo:
    gr.Markdown(f"# {APP_TITLE}\n{APP_DESCRIPTION}")
    with gr.Row():
        files_input = gr.File(label="Upload docs (optional: .md, .txt, .pdf)",
                              file_types=[".md", ".txt", ".pdf"], file_count="multiple", type="filepath")
        with gr.Column():
            status_md = gr.Markdown(f"Current documents: **{knowledge_base['name']}**")
            reload_btn = gr.Button("Reload my docs/ folder")
    question_box = gr.Textbox(label="Question", placeholder="For example: How do I submit a pull request?")
    with gr.Row():
        ask_btn = gr.Button("Ask", variant="primary")
        checklist_btn = gr.Button("Where do I start?")
    gr.Examples([
        "How do I set up the project on my computer?",
        "What is the process for submitting a pull request?",
        "Which coding style rules must I follow?",
        "What is the capital of France?",        # should be refused
        "Who won the 2022 World Cup?",           # should be refused
    ], inputs=question_box)  # STUDENT TODO: replace with questions for your chosen project
    answer_md = gr.Markdown()
    with gr.Accordion("Retrieved passages (the text given to Gemma)", open=True):
        sources_md = gr.Markdown()

    files_input.upload(on_upload, inputs=files_input, outputs=status_md)
    reload_btn.click(lambda: build_knowledge_base(SAMPLE_PAGES, PAGE_SOURCES, "Contributor docs"),
                     outputs=status_md)
    ask_btn.click(on_ask, inputs=question_box, outputs=[answer_md, sources_md])
    question_box.submit(on_ask, inputs=question_box, outputs=[answer_md, sources_md])
    checklist_btn.click(on_checklist, outputs=[answer_md, sources_md])

demo.launch(share=True)

Closing server running on port: 7860
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://122f01f442e9eeaa68.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Next steps
1. Re-read your selected problem statement (2A or 2B) in the Theme 2 document.
2. Replace the sample document and example questions with material for your problem statement.
3. Adapt `ANSWER_RULES`, the settings and the application to meet the requirements.
4. Measure the quality of the answers (see "Expected Deliverables" in the problem statement).
5. Attempt items from the **Good to have** section of the problem statement for additional credit.

**Pointers for common extensions:**
- **Markdown and text files (needed for 2B):** read them with `open(path, encoding="utf-8").read()` and treat each file (or each section under a heading) as one page, storing the file name for citations. Allow them in the application with `file_types=[".pdf", ".md", ".txt"]`.
- **Multiple documents:** use `gr.File(file_count="multiple")`, combine the pages of every file, and store the file name in each chunk for citations.
- **Answer without RAG (for comparison):** call `generate(question)` with no context and show the result next to the RAG answer.
- **Relevance threshold:** in `answer()`, if `found[0]["score"]` is below a chosen value, return "The answer could not be found in the document." without calling Gemma.
- **Conversation memory:** replace the question box with `gr.ChatInterface` and include recent messages in the prompt.

## Troubleshooting

| Issue | Resolution |
|---|---|
| The notebook remains in demonstration mode | Add `GOOGLE_API_KEY` in Secrets, enable **Notebook access**, and re-run the Step 2 cell. |
| "API key was rejected" | Create a new key at aistudio.google.com/apikey and update the secret. |
| "Usage limit reached" | Wait about one minute. Free keys allow a limited number of requests per minute. Team members may each create a key. |
| `NameError: name '...' is not defined` | A cell was skipped. Run all cells above in order. |
| "No text was found" | The PDF contains scanned images only. Use a text-based PDF. |
| The public link no longer works | Links expire when the Colab session disconnects. Re-run the launch cell to create a new link. |